<a href="https://colab.research.google.com/github/YukinoshitaSherry/CSCI572-Information_Retrieval_And_Web_Search_Engines/blob/master/scCode_claude_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch_geometric.nn import GATConv
import scanpy as sc
import anndata
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import optuna
from typing import Dict, List, Tuple, Optional, Union

# 设置随机种子以确保可重复性
def set_seed(seed: int = 42) -> None:
    """设置所有随机种子以确保实验可重复性"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# 1. 数据处理模块
class DataPreprocessor:
    """
    单细胞数据预处理模块，包括质控、标准化和批次校正
    """
    def __init__(self,
                 min_genes: int = 200,
                 min_cells: int = 3,
                 max_mt_percent: float = 10.0,
                 n_highly_variable: int = 2000):
        """
        初始化预处理参数

        参数:
            min_genes: 每个细胞最少基因数
            min_cells: 每个基因最少表达细胞数
            max_mt_percent: 线粒体基因最大百分比
            n_highly_variable: 高变异基因数量
        """
        self.min_genes = min_genes
        self.min_cells = min_cells
        self.max_mt_percent = max_mt_percent
        self.n_highly_variable = n_highly_variable

    def load_data(self, file_path: str) -> anndata.AnnData:
        """
        加载单细胞数据集

        参数:
            file_path: h5ad文件路径

        返回:
            anndata对象
        """
        print(f"Loading data from {file_path}")
        try:
            adata = sc.read_h5ad(file_path)
            print(f"Data loaded: {adata.shape[0]} cells, {adata.shape[1]} genes")
            return adata
        except Exception as e:
            print(f"Error loading data: {e}")
            raise

    def quality_control(self, adata: anndata.AnnData) -> anndata.AnnData:
        """
        执行单细胞数据质量控制

        参数:
            adata: 输入anndata对象

        返回:
            质控后的anndata对象
        """
        print("Performing quality control...")
        # 计算质控指标
        sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

        # 基于质控指标过滤细胞
        n_cells_before = adata.shape[0]
        adata = adata[adata.obs.n_genes_by_counts > self.min_genes]
        adata = adata[adata.obs.pct_counts_mt < self.max_mt_percent]

        # 过滤基因
        sc.pp.filter_genes(adata, min_cells=self.min_cells)

        n_cells_after = adata.shape[0]
        print(f"QC complete: {n_cells_before - n_cells_after} cells removed, {n_cells_after} cells remaining")
        return adata

    def normalize_and_scale(self, adata: anndata.AnnData) -> anndata.AnnData:
        """
        执行数据标准化和缩放

        参数:
            adata: 输入anndata对象

        返回:
            标准化后的anndata对象
        """
        print("Normalizing and scaling data...")
        # 使用SCTransform方法进行标准化（这里使用scanpy的标准标准化作为替代）
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # 选择高变异基因
        sc.pp.highly_variable_genes(adata, n_top_genes=self.n_highly_variable)
        adata = adata[:, adata.var.highly_variable]

        print(f"Normalization complete: {adata.shape[1]} highly variable genes selected")
        return adata

    def batch_correction(self, adata: anndata.AnnData, batch_key: str = 'batch') -> anndata.AnnData:
        """
        执行批次校正

        参数:
            adata: 输入anndata对象
            batch_key: 批次信息在obs中的键名

        返回:
            批次校正后的anndata对象
        """
        print("Performing batch correction...")
        if batch_key in adata.obs.columns:
            # 使用Harmony进行批次校正
            # 注意：这里我们使用scanpy的PCA+Neighbors作为替代，实际项目中应该使用harmony_pytorch
            sc.pp.pca(adata)
            sc.pp.neighbors(adata)
            print("Batch correction complete")
        else:
            print(f"No batch correction performed: '{batch_key}' not found in adata.obs")

        return adata

    def process_data(self, file_path: str, batch_key: Optional[str] = None) -> anndata.AnnData:
        """
        执行完整的数据预处理流程

        参数:
            file_path: h5ad文件路径
            batch_key: 批次信息键名，如果为None则不进行批次校正

        返回:
            处理后的anndata对象
        """
        adata = self.load_data(file_path)
        adata = self.quality_control(adata)
        adata = self.normalize_and_scale(adata)

        if batch_key is not None:
            adata = self.batch_correction(adata, batch_key)

        return adata


In [ ]:
# 2. 多模态数据集成模块 - 分层变分自编码器 (Hierarchical VAE)
class HVAE(nn.Module):
    """
    分层变分自编码器，用于多模态数据集成
    """
    def __init__(self,
                 input_dims: Dict[str, int],
                 latent_dim: int = 32,
                 hidden_dims: List[int] = [128, 64],
                 l2_reg: float = 1e-5):
        """
        初始化HVAE模型

        参数:
            input_dims: 各模态输入维度的字典，例如 {'rna': 2000, 'protein': 100}
            latent_dim: 潜在空间维度
            hidden_dims: 隐藏层维度列表
            l2_reg: L2正则化强度
        """
        super(HVAE, self).__init__()
        self.input_dims = input_dims
        self.latent_dim = latent_dim
        self.hidden_dims = hidden_dims
        self.l2_reg = l2_reg

        # 为每个模态创建编码器
        self.encoders = nn.ModuleDict()
        self.mu_projectors = nn.ModuleDict()
        self.logvar_projectors = nn.ModuleDict()

        for modality, dim in input_dims.items():
            encoder_layers = []
            input_dim = dim

            # 构建编码器层
            for hidden_dim in hidden_dims:
                encoder_layers.append(nn.Linear(input_dim, hidden_dim))
                encoder_layers.append(nn.BatchNorm1d(hidden_dim))
                encoder_layers.append(nn.LeakyReLU())
                encoder_layers.append(nn.Dropout(0.2))
                input_dim = hidden_dim

            self.encoders[modality] = nn.Sequential(*encoder_layers)

            # 均值和方差投影器
            self.mu_projectors[modality] = nn.Linear(hidden_dims[-1], latent_dim)
            self.logvar_projectors[modality] = nn.Linear(hidden_dims[-1], latent_dim)

        # 联合潜在空间投影器
        self.joint_encoder = nn.Sequential(
            nn.Linear(latent_dim * len(input_dims), hidden_dims[-1]),
            nn.BatchNorm1d(hidden_dims[-1]),
            nn.LeakyReLU(),
            nn.Dropout(0.2)
        )

        self.joint_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.joint_logvar = nn.Linear(hidden_dims[-1], latent_dim)

        # 解码器 - 从联合潜在空间到各模态
        self.decoders = nn.ModuleDict()

        for modality, dim in input_dims.items():
            decoder_layers = []
            input_dim = latent_dim

            # 构建解码器层
            for hidden_dim in reversed(hidden_dims):
                decoder_layers.append(nn.Linear(input_dim, hidden_dim))
                decoder_layers.append(nn.BatchNorm1d(hidden_dim))
                decoder_layers.append(nn.LeakyReLU())
                decoder_layers.append(nn.Dropout(0.2))
                input_dim = hidden_dim

            # 输出层
            decoder_layers.append(nn.Linear(hidden_dims[0], dim))

            self.decoders[modality] = nn.Sequential(*decoder_layers)

    def encode(self, x_dict: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        编码多模态输入到联合潜在空间

        参数:
            x_dict: 各模态输入数据的字典

        返回:
            mu: 联合潜在空间的均值
            logvar: 联合潜在空间的对数方差
        """
        modal_latents = []

        # 编码每个模态
        for modality, encoder in self.encoders.items():
            if modality in x_dict:
                x = x_dict[modality]
                hidden = encoder(x)
                mu = self.mu_projectors[modality](hidden)
                logvar = self.logvar_projectors[modality](hidden)
                z = self.reparameterize(mu, logvar)
                modal_latents.append(z)
            else:
                # 如果某个模态缺失，用零向量代替
                batch_size = next(iter(x_dict.values())).size(0)
                modal_latents.append(torch.zeros(batch_size, self.latent_dim, device=next(iter(x_dict.values())).device))

        # 将所有模态的潜在表示连接起来
        z_concat = torch.cat(modal_latents, dim=1)

        # 联合编码
        joint_hidden = self.joint_encoder(z_concat)
        joint_mu = self.joint_mu(joint_hidden)
        joint_logvar = self.joint_logvar(joint_hidden)

        return joint_mu, joint_logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """
        重参数化技巧，使得梯度可以通过随机采样操作传播

        参数:
            mu: 均值
            logvar: 对数方差

        返回:
            采样的潜在向量
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        从联合潜在空间解码到各模态

        参数:
            z: 潜在向量

        返回:
            各模态重构输出的字典
        """
        reconstructions = {}

        for modality, decoder in self.decoders.items():
            reconstructions[modality] = decoder(z)

        return reconstructions

    def forward(self, x_dict: Dict[str, torch.Tensor]) -> Tuple[Dict[str, torch.Tensor], torch.Tensor, torch.Tensor]:
        """
        前向传播

        参数:
            x_dict: 各模态输入数据的字典

        返回:
            reconstructions: 各模态重构输出的字典
            mu: 联合潜在空间的均值
            logvar: 联合潜在空间的对数方差
        """
        mu, logvar = self.encode(x_dict)
        z = self.reparameterize(mu, logvar)
        reconstructions = self.decode(z)

        return reconstructions, mu, logvar

    def loss_function(self,
                      x_dict: Dict[str, torch.Tensor],
                      reconstructions: Dict[str, torch.Tensor],
                      mu: torch.Tensor,
                      logvar: torch.Tensor,
                      kld_weight: float = 0.005) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        计算VAE损失函数

        参数:
            x_dict: 各模态原始输入的字典
            reconstructions: 各模态重构输出的字典
            mu: 均值
            logvar: 对数方差
            kld_weight: KL散度权重

        返回:
            total_loss: 总损失
            loss_dict: 各部分损失的字典，用于监控
        """
        recon_losses = {}

        # 计算各模态的重构损失
        for modality in x_dict.keys():
            recon_x = reconstructions[modality]
            x = x_dict[modality]

            # 使用均方误差作为重构损失
            recon_losses[modality] = F.mse_loss(recon_x, x, reduction='sum') / x.size(0)

        # 总重构损失 - 各模态重构损失的和
        recon_loss = sum(recon_losses.values())

        # KL散度
        kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.size(0)

        # L2正则化损失
        l2_reg_loss = 0
        for param in self.parameters():
            l2_reg_loss += torch.norm(param, p=2)
        l2_reg_loss *= self.l2_reg

        # 总损失
        total_loss = recon_loss + kld_weight * kld_loss + l2_reg_loss

        # 创建损失字典用于监控
        loss_dict = {
            'total_loss': total_loss.item(),
            'recon_loss': recon_loss.item(),
            'kld_loss': kld_loss.item(),
            'l2_reg_loss': l2_reg_loss.item()
        }

        # 添加各模态的重构损失
        for modality, loss in recon_losses.items():
            loss_dict[f'recon_loss_{modality}'] = loss.item()

        return total_loss, loss_dict



In [ ]:
# 3. 通路级建模模块 - 图注意力网络 (GAT)
class PathwayGNN(nn.Module):
    """
    基于图注意力网络的通路级模型
    """
    def __init__(self,
                 input_dim: int,
                 hidden_dim: int = 64,
                 output_dim: int = 32,
                 n_heads: int = 4,
                 dropout: float = 0.2,
                 alpha: float = 0.2):
        """
        初始化通路GNN模型

        参数:
            input_dim: 输入特征维度
            hidden_dim: 隐藏层维度
            output_dim: 输出特征维度
            n_heads: 注意力头数量
            dropout: Dropout概率
            alpha: LeakyReLU负斜率
        """
        super(PathwayGNN, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.n_heads = n_heads
        self.dropout = dropout

        # 第一层GAT，多头注意力
        self.gat1 = GATConv(
            in_channels=input_dim,
            out_channels=hidden_dim // n_heads,
            heads=n_heads,
            dropout=dropout,
            negative_slope=alpha
        )

        # 第二层GAT，单头输出
        self.gat2 = GATConv(
            in_channels=hidden_dim,
            out_channels=output_dim,
            heads=1,
            concat=False,
            dropout=dropout,
            negative_slope=alpha
        )

        # 特征转换层
        self.transform = nn.Sequential(
            nn.Linear(output_dim, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.LeakyReLU(alpha),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        前向传播

        参数:
            x: 节点特征矩阵，形状为 [num_nodes, input_dim]
            edge_index: 边索引，形状为 [2, num_edges]

        返回:
            节点嵌入，形状为 [num_nodes, output_dim]
        """
        # 第一层GAT
        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # 第二层GAT
        x = self.gat2(x, edge_index)

        # 特征转换
        x = self.transform(x)

        return x

    def get_attention_weights(self, x: torch.Tensor, edge_index: torch.Tensor) -> List[torch.Tensor]:
        """
        获取注意力权重

        参数:
            x: 节点特征矩阵
            edge_index: 边索引

        返回:
            各层注意力权重列表
        """
        # 获取第一层GAT的注意力权重
        _, attn_weights_1 = self.gat1(x, edge_index, return_attention_weights=True)

        # 获取第二层GAT的注意力权重
        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        _, attn_weights_2 = self.gat2(x, edge_index, return_attention_weights=True)

        return [attn_weights_1, attn_weights_2]



In [ ]:
# 4. 时序动态模块 - LSTM
class TemporalLSTM(nn.Module):
    """
    用于捕获T细胞响应时间动态的LSTM模型
    """
    def __init__(self,
                 input_dim: int,
                 hidden_dim: int = 64,
                 num_layers: int = 2,
                 output_dim: int = 32,
                 dropout: float = 0.2,
                 bidirectional: bool = True):
        """
        初始化时序LSTM模型

        参数:
            input_dim: 输入特征维度
            hidden_dim: LSTM隐藏层维度
            num_layers: LSTM层数
            output_dim: 输出特征维度
            dropout: Dropout概率
            bidirectional: 是否使用双向LSTM
        """
        super(TemporalLSTM, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.output_dim = output_dim
        self.bidirectional = bidirectional

        # LSTM层
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )

        # 输出投影层
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.output_projection = nn.Sequential(
            nn.Linear(lstm_output_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x: torch.Tensor, lengths: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        前向传播

        参数:
            x: 输入序列，形状为 [batch_size, seq_len, input_dim]
            lengths: 序列实际长度，用于pack_padded_sequence，形状为 [batch_size]

        返回:
            输出特征，形状为 [batch_size, output_dim]
        """
        batch_size, seq_len, _ = x.size()

        if lengths is not None:
            # 打包填充序列以提高效率
            x_packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)

            # 通过LSTM
            output_packed, (hidden, _) = self.lstm(x_packed)

            # 解包序列
            output, _ = nn.utils.rnn.pad_packed_sequence(output_packed, batch_first=True)
        else:
            # 不使用序列长度信息
            output, (hidden, _) = self.lstm(x)

        # 使用最后一个非填充时间步的输出
        if self.bidirectional:
            # 对于双向LSTM，合并最后一层的前向和后向隐藏状态
            last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            # 对于单向LSTM，使用最后一层的隐藏状态
            last_hidden = hidden[-1]

        # 通过输出投影层
        output = self.output_projection(last_hidden)

        return output


In [ ]:

# 5. 扰动预测模块
class PerturbationPredictor(nn.Module):
    """
    T细胞扰动响应预测器，集成HVAE、PathwayGNN和TemporalLSTM
    """
    def __init__(self,
                 hvae: HVAE,
                 pathway_gnn: PathwayGNN,
                 temporal_lstm: TemporalLSTM,
                 latent_dim: int,
                 num_perturbations: int,
                 pathway_nodes: int,
                 time_points: int):
        """
        初始化扰动预测器

        参数:
            hvae: 预训练的HVAE模型
            pathway_gnn: 预训练的PathwayGNN模型
            temporal_lstm: 预训练的TemporalLSTM模型
            latent_dim: 潜在空间维度
            num_perturbations: 扰动类型数量
            pathway_nodes: 通路图中的节点数量
            time_points: 时间点数量
        """
        super(PerturbationPredictor, self).__init__()
        self.hvae = hvae
        self.pathway_gnn = pathway_gnn
        self.temporal_lstm = temporal_lstm
        self.latent_dim = latent_dim
        self.num_perturbations = num_perturbations
        self.pathway_nodes = pathway_nodes
        self.time_points = time_points

        # 冻结HVAE权重
        for param in self.hvae.parameters():
            param.requires_grad = False

        # 扰动嵌入层
        self.perturbation_embedding = nn.Embedding(num_perturbations, latent_dim)

        # 时间嵌入层
        self.time_embedding = nn.Embedding(time_points, latent_dim)

        # 融合层
        self.fusion_layer = nn.Sequential(
            nn.Linear(latent_dim * 3, latent_dim * 2),
            nn.LayerNorm(latent_dim * 2),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(latent_dim * 2, latent_dim)
        )

        # 预测头 - 预测细胞状态变化
        self.prediction_head = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(latent_dim, pathway_nodes)  # 预测每个通路节点的响应
        )

    def forward(self,
                x_dict: Dict[str, torch.Tensor],
                perturbation_ids: torch.Tensor,
                time_ids: torch.Tensor,
                edge_index: torch.Tensor) -> torch.Tensor:
        """
        前向传播

        参数:
            x_dict: 各模态输入数据的字典
            perturbation_ids: 扰动类型ID，形状为 [batch_size]
            time_ids: 时间点ID，形状为 [batch_size]
            edge_index: 通路图的边索引

        返回:
            预测的通路响应，形状为 [batch_size, pathway_nodes]
        """
        # 从HVAE获取细胞状态表示
        with torch.no_grad():
            _, mu, _ = self.hvae(x_dict)
            cell_state = mu

        # 获取扰动嵌入
        perturbation_emb = self.perturbation_embedding(perturbation_ids)

        # 获取时间嵌入
        time_emb = self.time_embedding(time_ids)

        # 融合细胞状态、扰动和时间信息
        fusion_input = torch.cat([cell_state, perturbation_emb, time_emb], dim=1)
        fused_representation = self.fusion_layer(fusion_input)

        # 生成通路图的节点表示
        # 假设我们有一个映射函数，将融合表示映射到通路图的节点特征
        # 这里简化处理，将融合表示复制到每个节点
        node_features = fused_representation.unsqueeze(1).expand(-1, self.pathway_nodes, -1)
        node_features = node_features.reshape(-1, self.latent_dim)

        # 使用PathwayGNN更新节点表示
        node_embeddings = self.pathway_gnn(node_features, edge_index)

        # 重新整形为[batch_size, pathway_nodes, embedding_dim]
        node_embeddings = node_embeddings.view(-1, self.pathway_nodes, self.pathway_gnn.output_dim)

        # 使用TemporalLSTM捕获时序动态
        # 简化处理，假设我们使用单个时间点，不使用序列信息
        temporal_features = node_embeddings.mean(dim=1)  # 对所有节点取平均
        temporal_output = self.temporal_lstm(temporal_features.unsqueeze(1))

        # 最终预测
        predictions = self.prediction_head(temporal_output)

        return predictions


In [ ]:

# 6. 模型训练与评估
class ModelTrainer:
    """
    模型训练与评估管理类
    """
    def __init__(self,
                 device: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
                 experiment_name: str = 'tcell_perturbation_model',
                 log_dir: str = './logs',
                 checkpoint_dir: str = './checkpoints'):
        """
        初始化模型训练器

        参数:
            device: 训练设备
            experiment_name: 实验名称
            log_dir: 日志目录
            checkpoint_dir: 检查点目录
        """
        self.device = device
        self.experiment_name = experiment_name
        self.log_dir = log_dir
        self.checkpoint_dir = checkpoint_dir

        # 创建必要的目录
        os.makedirs(log_dir, exist_ok=True)
        os.makedirs(checkpoint_dir, exist_ok=True)

        # 初始化MLflow跟踪
        mlflow.set_experiment(experiment_name)
        print(f"Using device: {device}")

    def train_hvae(self,
                   hvae: HVAE,
                   train_loader: DataLoader,
                   val_loader: DataLoader,
                   optimizer: torch.optim.Optimizer,
                   num_epochs: int = 50,
                   kld_weight: float = 0.005,
                   early_stopping_patience: int = 10,
                   grad_clip_val: float = 1.0) -> HVAE:
        """
        训练HVAE模型

        参数:
            hvae: HVAE模型
            train_loader: 训练数据加载器
            val_loader: 验证数据加载器
            optimizer: 优化器
            num_epochs: 训练轮数
            kld_weight: KL散度权重
            early_stopping_patience: 早停耐心值
            grad_clip_val: 梯度裁剪值

        返回:
            训练好的HVAE模型
        """
        print(f"Training HVAE model for {num_epochs} epochs...")
        hvae.to(self.device)

        # 早停
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None

        with mlflow.start_run(run_name='hvae_training'):
            mlflow.log_param('model_type', 'HVAE')
            mlflow.log_param('latent_dim', hvae.latent_dim)
            mlflow.log_param('hidden_dims', hvae.hidden_dims)
            mlflow.log_param('l2_reg', hvae.l2_reg)
            mlflow.log_param('kld_weight', kld_weight)
            mlflow.log_param('grad_clip_val', grad_clip_val)

            for epoch in range(num_epochs):
                # 训练模式
                hvae.train()
                train_loss = 0
                train_loss_dict = {k: 0 for k in ['total_loss', 'recon_loss', 'kld_loss', 'l2_reg_loss']}

                for batch_idx, batch in enumerate(train_loader):
                    # 将数据移到设备
                    x_dict = {k: v.to(self.device) for k, v in batch.items() if k in hvae.input_dims}

                    # 前向传播
                    optimizer.zero_grad()
                    reconstructions, mu, logvar = hvae(x_dict)
                    loss, loss_dict = hvae.loss_function(x_dict, reconstructions, mu, logvar, kld_weight)

                    # 反向传播
                    loss.backward()

                    # 梯度裁剪
                    nn.utils.clip_grad_norm_(hvae.parameters(), grad_clip_val)

                    optimizer.step()

                    # 累计损失
                    train_loss += loss.item()
                    for k, v in loss_dict.items():
                        train_loss_dict[k] += v

                # 计算平均训练损失
                train_loss /= len(train_loader)
                for k in train_loss_dict:
                    train_loss_dict[k] /= len(train_loader)

                # 验证模式
                hvae.eval()
                val_loss = 0
                val_loss_dict = {k: 0 for k in ['total_loss', 'recon_loss', 'kld_loss', 'l2_reg_loss']}

                with torch.no_grad():
                    for batch_idx, batch in enumerate(val_loader):
                        # 将数据移到设备
                        x_dict = {k: v.to(self.device) for k, v in batch.items() if k in hvae.input_dims}

                        # 前向传播
                        reconstructions, mu, logvar = hvae(x_dict)
                        loss, loss_dict = hvae.loss_function(x_dict, reconstructions, mu, logvar, kld_weight)

                        # 累计损失
                        val_loss += loss.item()
                        for k, v in loss_dict.items():
                            val_loss_dict[k] += v

                # 计算平均验证损失
                val_loss /= len(val_loader)
                for k in val_loss_dict:
                    val_loss_dict[k] /= len(val_loader)

                # 打印进度
                print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

                # 记录指标
                mlflow.log_metric('train_loss', train_loss, step=epoch)
                mlflow.log_metric('val_loss', val_loss, step=epoch)
                for k, v in train_loss_dict.items():
                    mlflow.log_metric(f'train_{k}', v, step=epoch)
                for k, v in val_loss_dict.items():
                    mlflow.log_metric(f'val_{k}', v, step=epoch)

                # 检查早停
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    best_model_state = hvae.state_dict().copy()

                    # 保存最佳模型
                    checkpoint_path = os.path.join(self.checkpoint_dir, f'hvae_best.pt')
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': hvae.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'val_loss': val_loss,
                    }, checkpoint_path)
                    mlflow.log_artifact(checkpoint_path)
                else:
                    patience_counter += 1
                    if patience_counter >= early_stopping_patience:
                        print(f'Early stopping triggered after {epoch+1} epochs')
                        break

            # 保存最终模型
            checkpoint_path = os.path.join(self.checkpoint_dir, f'hvae_final.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': hvae.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, checkpoint_path)
            mlflow.log_artifact(checkpoint_path)

        # 加载最佳模型状态
        if best_model_state is not None:
            hvae.load_state_dict(best_model_state)

        return hvae

    def train_pathway_gnn(self,
                          pathway_gnn: PathwayGNN,
                          train_loader: DataLoader,
                          val_loader: DataLoader,
                          optimizer: torch.optim.Optimizer,
                          loss_fn: callable,
                          num_epochs: int = 50,
                          early_stopping_patience: int = 10,
                          grad_clip_val: float = 1.0) -> PathwayGNN:
        """
        训练PathwayGNN模型

        参数:
            pathway_gnn: PathwayGNN模型
            train_loader: 训练数据加载器
            val_loader: 验证数据加载器
            optimizer: 优化器
            loss_fn: 损失函数
            num_epochs: 训练轮数
            early_stopping_patience: 早停耐心值
            grad_clip_val: 梯度裁剪值

        返回:
            训练好的PathwayGNN模型
        """
        print(f"Training PathwayGNN model for {num_epochs} epochs...")
        pathway_gnn.to(self.device)

        # 早停
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None

        with mlflow.start_run(run_name='pathway_gnn_training'):
            mlflow.log_param('model_type', 'PathwayGNN')
            mlflow.log_param('input_dim', pathway_gnn.input_dim)
            mlflow.log_param('hidden_dim', pathway_gnn.hidden_dim)
            mlflow.log_param('output_dim', pathway_gnn.output_dim)
            mlflow.log_param('n_heads', pathway_gnn.n_heads)
            mlflow.log_param('dropout', pathway_gnn.dropout)
            mlflow.log_param('grad_clip_val', grad_clip_val)

            for epoch in range(num_epochs):
                # 训练模式
                pathway_gnn.train()
                train_loss = 0

                for batch_idx, batch in enumerate(train_loader):
                    # 将数据移到设备
                    x, edge_index, y = batch
                    x = x.to(self.device)
                    edge_index = edge_index.to(self.device)
                    y = y.to(self.device)

                    # 前向传播
                    optimizer.zero_grad()
                    output = pathway_gnn(x, edge_index)
                    loss = loss_fn(output, y)

                    # 反向传播
                    loss.backward()

                    # 梯度裁剪
                    nn.utils.clip_grad_norm_(pathway_gnn.parameters(), grad_clip_val)

                    optimizer.step()

                    # 累计损失
                    train_loss += loss.item()

                # 计算平均训练损失
                train_loss /= len(train_loader)

                # 验证模式
                pathway_gnn.eval()
                val_loss = 0

                with torch.no_grad():
                    for batch_idx, batch in enumerate(val_loader):
                        # 将数据移到设备
                        x, edge_index, y = batch
                        x = x.to(self.device)
                        edge_index = edge_index.to(self.device)
                        y = y.to(self.device)

                        # 前向传播
                        output = pathway_gnn(x, edge_index)
                        loss = loss_fn(output, y)

                        # 累计损失
                        val_loss += loss.item()

                # 计算平均验证损失
                val_loss /= len(val_loader)

                # 打印进度
                print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

                # 记录指标
                mlflow.log_metric('train_loss', train_loss, step=epoch)
                mlflow.log_metric('val_loss', val_loss, step=epoch)

                # 检查早停
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    best_model_state = pathway_gnn.state_dict().copy()

                    # 保存最佳模型
                    checkpoint_path = os.path.join(self.checkpoint_dir, f'pathway_gnn_best.pt')
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': pathway_gnn.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'val_loss': val_loss,
                    }, checkpoint_path)
                    mlflow.log_artifact(checkpoint_path)
                else:
                    patience_counter += 1
                    if patience_counter >= early_stopping_patience:
                        print(f'Early stopping triggered after {epoch+1} epochs')
                        break

            # 保存最终模型
            checkpoint_path = os.path.join(self.checkpoint_dir, f'pathway_gnn_final.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': pathway_gnn.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, checkpoint_path)
            mlflow.log_artifact(checkpoint_path)

        # 加载最佳模型状态
        if best_model_state is not None:
            pathway_gnn.load_state_dict(best_model_state)

        return pathway_gnn

    def train_temporal_lstm(self,
                           temporal_lstm: TemporalLSTM,
                           train_loader: DataLoader,
                           val_loader: DataLoader,
                           optimizer: torch.optim.Optimizer,
                           loss_fn: callable,
                           num_epochs: int = 50,
                           early_stopping_patience: int = 10,
                           grad_clip_val: float = 1.0) -> TemporalLSTM:
        """
        训练TemporalLSTM模型

        参数:
            temporal_lstm: TemporalLSTM模型
            train_loader: 训练数据加载器
            val_loader: 验证数据加载器
            optimizer: 优化器
            loss_fn: 损失函数
            num_epochs: 训练轮数
            early_stopping_patience: 早停耐心值
            grad_clip_val: 梯度裁剪值

        返回:
            训练好的TemporalLSTM模型
        """
        print(f"Training TemporalLSTM model for {num_epochs} epochs...")
        temporal_lstm.to(self.device)

        # 早停
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None

        with mlflow.start_run(run_name='temporal_lstm_training'):
            mlflow.log_param('model_type', 'TemporalLSTM')
            mlflow.log_param('input_dim', temporal_lstm.input_dim)
            mlflow.log_param('hidden_dim', temporal_lstm.hidden_dim)
            mlflow.log_param('num_layers', temporal_lstm.num_layers)
            mlflow.log_param('output_dim', temporal_lstm.output_dim)
            mlflow.log_param('bidirectional', temporal_lstm.bidirectional)
            mlflow.log_param('grad_clip_val', grad_clip_val)

            for epoch in range(num_epochs):
                # 训练模式
                temporal_lstm.train()
                train_loss = 0

                for batch_idx, batch in enumerate(train_loader):
                    # 将数据移到设备
                    x, lengths, y = batch
                    x = x.to(self.device)
                    if lengths is not None:
                        lengths = lengths.to(self.device)
                    y = y.to(self.device)

                    # 前向传播
                    optimizer.zero_grad()
                    output = temporal_lstm(x, lengths)
                    loss = loss_fn(output, y)

                    # 反向传播
                    loss.backward()

                    # 梯度裁剪
                    nn.utils.clip_grad_norm_(temporal_lstm.parameters(), grad_clip_val)

                    optimizer.step()

                    # 累计损失
                    train_loss += loss.item()

                # 计算平均训练损失
                train_loss /= len(train_loader)

                # 验证模式
                temporal_lstm.eval()
                val_loss = 0

                with torch.no_grad():
                    for batch_idx, batch in enumerate(val_loader):
                        # 将数据移到设备
                        x, lengths, y = batch
                        x = x.to(self.device)
                        if lengths is not None:
                            lengths = lengths.to(self.device)
                        y = y.to(self.device)

                        # 前向传播
                        output = temporal_lstm(x, lengths)
                        loss = loss_fn(output, y)

                        # 累计损失
                        val_loss += loss.item()

                # 计算平均验证损失
                val_loss /= len(val_loader)

                # 打印进度
                print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

                # 记录指标
                mlflow.log_metric('train_loss', train_loss, step=epoch)
                mlflow.log_metric('val_loss', val_loss, step=epoch)

                # 检查早停
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    best_model_state = temporal_lstm.state_dict().copy()

                    # 保存最佳模型
                    checkpoint_path = os.path.join(self.checkpoint_dir, f'temporal_lstm_best.pt')
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': temporal_lstm.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'val_loss': val_loss,
                    }, checkpoint_path)
                    mlflow.log_artifact(checkpoint_path)
                else:
                    patience_counter += 1
                    if patience_counter >= early_stopping_patience:
                        print(f'Early stopping triggered after {epoch+1} epochs')
                        break

            # 保存最终模型
            checkpoint_path = os.path.join(self.checkpoint_dir, f'temporal_lstm_final.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': temporal_lstm.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, checkpoint_path)
            mlflow.log_artifact(checkpoint_path)

        # 加载最佳模型状态
        if best_model_state is not None:
            temporal_lstm.load_state_dict(best_model_state)

        return temporal_lstm

    def train_perturbation_predictor(self,
                                   predictor: PerturbationPredictor,
                                   train_loader: DataLoader,
                                   val_loader: DataLoader,
                                   optimizer: torch.optim.Optimizer,
                                   loss_fn: callable,
                                   num_epochs: int = 50,
                                   early_stopping_patience: int = 10,
                                   grad_clip_val: float = 1.0) -> PerturbationPredictor:
        """
        训练扰动预测器模型

        参数:
            predictor: PerturbationPredictor模型
            train_loader: 训练数据加载器
            val_loader: 验证数据加载器
            optimizer: 优化器
            loss_fn: 损失函数
            num_epochs: 训练轮数
            early_stopping_patience: 早停耐心值
            grad_clip_val: 梯度裁剪值

        返回:
            训练好的PerturbationPredictor模型
        """
        print(f"Training PerturbationPredictor model for {num_epochs} epochs...")
        predictor.to(self.device)

        # 早停
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None

        with mlflow.start_run(run_name='perturbation_predictor_training'):
            mlflow.log_param('model_type', 'PerturbationPredictor')
            mlflow.log_param('latent_dim', predictor.latent_dim)
            mlflow.log_param('num_perturbations', predictor.num_perturbations)
            mlflow.log_param('pathway_nodes', predictor.pathway_nodes)
            mlflow.log_param('time_points', predictor.time_points)
            mlflow.log_param('grad_clip_val', grad_clip_val)

            for epoch in range(num_epochs):
                # 训练模式
                predictor.train()
                train_loss = 0

                for batch_idx, batch in enumerate(train_loader):
                    # 将数据移到设备
                    x_dict, perturbation_ids, time_ids, edge_index, y = batch
                    x_dict = {k: v.to(self.device) for k, v in x_dict.items()}
                    perturbation_ids = perturbation_ids.to(self.device)
                    time_ids = time_ids.to(self.device)
                    edge_index = edge_index.to(self.device)
                    y = y.to(self.device)

                    # 前向传播
                    optimizer.zero_grad()
                    output = predictor(x_dict, perturbation_ids, time_ids, edge_index)
                    loss = loss_fn(output, y)

                    # 反向传播
                    loss.backward()

                    # 梯度裁剪
                    nn.utils.clip_grad_norm_(predictor.parameters(), grad_clip_val)

                    optimizer.step()

                    # 累计损失
                    train_loss += loss.item()

                # 计算平均训练损失
                train_loss /= len(train_loader)

                # 验证模式
                predictor.eval()
                val_loss = 0
                y_true_all = []
                y_pred_all = []

                with torch.no_grad():
                    for batch_idx, batch in enumerate(val_loader):
                        # 将数据移到设备
                        x_dict, perturbation_ids, time_ids, edge_index, y = batch
                        x_dict = {k: v.to(self.device) for k, v in x_dict.items()}
                        perturbation_ids = perturbation_ids.to(self.device)
                        time_ids = time_ids.to(self.device)
                        edge_index = edge_index.to(self.device)
                        y = y.to(self.device)

                        # 前向传播
                        output = predictor(x_dict, perturbation_ids, time_ids, edge_index)
                        loss = loss_fn(output, y)

                        # 累计损失
                        val_loss += loss.item()

                        # 收集预测和真实值用于计算指标
                        y_true_all.append(y.cpu().numpy())
                        y_pred_all.append(output.cpu().numpy())

                # 计算平均验证损失
                val_loss /= len(val_loader)

                # 计算验证指标
                y_true = np.concatenate(y_true_all)
                y_pred = np.concatenate(y_pred_all)

                # 计算每个通路节点的AUC（简化处理：二值化预测，阈值为0.5）
                aucs = []
                for i in range(y_true.shape[1]):
                    if len(np.unique(y_true[:, i])) > 1:  # 确保有正负样本
                        auc_score = roc_auc_score(y_true[:, i] > 0.5, y_pred[:, i])
                        aucs.append(auc_score)

                mean_auc = np.mean(aucs) if aucs else 0.0

                # 打印进度
                print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Mean AUC: {mean_auc:.4f}')

                # 记录指标
                mlflow.log_metric('train_loss', train_loss, step=epoch)
                mlflow.log_metric('val_loss', val_loss, step=epoch)
                mlflow.log_metric('mean_auc', mean_auc, step=epoch)

                # 检查早停
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    best_model_state = predictor.state_dict().copy()

                    # 保存最佳模型
                    checkpoint_path = os.path.join(self.checkpoint_dir, f'perturbation_predictor_best.pt')
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': predictor.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'val_loss': val_loss,
                        'mean_auc': mean_auc,
                    }, checkpoint_path)
                    mlflow.log_artifact(checkpoint_path)
                else:
                    patience_counter += 1
                    if patience_counter >= early_stopping_patience:
                        print(f'Early stopping triggered after {epoch+1} epochs')
                        break

            # 保存最终模型
            checkpoint_path = os.path.join(self.checkpoint_dir, f'perturbation_predictor_final.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': predictor.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'mean_auc': mean_auc,
            }, checkpoint_path)
            mlflow.log_artifact(checkpoint_path)

        # 加载最佳模型状态
        if best_model_state is not None:
            predictor.load_state_dict(best_model_state)

        return predictor

    def evaluate_model(self,
                      predictor: PerturbationPredictor,
                      test_loader: DataLoader,
                      loss_fn: callable) -> Dict[str, float]:
        """
        在测试集上评估模型

        参数:
            predictor: 训练好的PerturbationPredictor模型
            test_loader: 测试数据加载器
            loss_fn: 损失函数

        返回:
            评估指标字典
        """
        print("Evaluating model on test set...")
        predictor.to(self.device)
        predictor.eval()

        test_loss = 0
        y_true_all = []
        y_pred_all = []

        with torch.no_grad():
            for batch_idx, batch in enumerate(test_loader):
                # 将数据移到设备
                x_dict, perturbation_ids, time_ids, edge_index, y = batch
                x_dict = {k: v.to(self.device) for k, v in x_dict.items()}
                perturbation_ids = perturbation_ids.to(self.device)
                time_ids = time_ids.to(self.device)
                edge_index = edge_index.to(self.device)
                y = y.to(self.device)

                # 前向传播
                output = predictor(x_dict, perturbation_ids, time_ids, edge_index)
                loss = loss_fn(output, y)

                # 累计损失
                test_loss += loss.item()

                # 收集预测和真实值
                y_true_all.append(y.cpu().numpy())
                y_pred_all.append(output.cpu().numpy())

        # 计算平均测试损失
        test_loss /= len(test_loader)

        # 计算其他指标
        y_true = np.concatenate(y_true_all)
        y_pred = np.concatenate(y_pred_all)

        # 计算每个通路节点的AUC和PR AUC
        aucs = []
        pr_aucs = []
        for i in range(y_true.shape[1]):
            if len(np.unique(y_true[:, i])) > 1:  # 确保有正负样本
                # ROC AUC
                auc_score = roc_auc_score(y_true[:, i] > 0.5, y_pred[:, i])
                aucs.append(auc_score)

                # PR AUC
                precision, recall, _ = precision_recall_curve(y_true[:, i] > 0.5, y_pred[:, i])
                pr_auc = auc(recall, precision)
                pr_aucs.append(pr_auc)

        mean_auc = np.mean(aucs) if aucs else 0.0
        mean_pr_auc = np.mean(pr_aucs) if pr_aucs else 0.0

        # 返回评估指标
        metrics = {
            'test_loss': test_loss,
            'mean_auc': mean_auc,
            'mean_pr_auc': mean_pr_auc,
            'node_aucs': aucs,
            'node_pr_aucs': pr_aucs
        }

        # 记录评估结果
        with mlflow.start_run(run_name='model_evaluation'):
            mlflow.log_metrics(metrics)
